In [1]:
import os
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Noul, NoulCriteria

load_dotenv(override=True)

client = TypeSafeClient(
    api_key=os.environ["TYPESAFE_API_KEY"],
    timeout=120.0
)

def should_merge(text_a, text_b):

    state = f"""
TEXT A:
{text_a}

TEXT B:
{text_b}
"""

    response = client.system_one(
        model="jev-latest",
        state=state,
        questions={
            "same_chunk": Noul(
                instructions=(
                    "Should these two pieces of text belong "
                    "to the same semantic chunk?"
                ),
                criteria=NoulCriteria(
                    true=(
                        "The two pieces discuss the same coherent topic "
                        "and keeping them together preserves useful context."
                    ),
                    false=(
                        "The two pieces discuss different topics or "
                        "represent a meaningful semantic boundary."
                    )
                )
            )
        }
    )

    return response.answers["same_chunk"].noul



In [ ]:
# Test 1
text_a = """
Insurellm offers health insurance to all full-time employees.
"""

text_b = """
Employees become eligible for health insurance after 30 days
of employment.
"""

probability = should_merge(text_a, text_b)

print("Probability of same chunk:", probability)

In [2]:
# Test 2
text_a = """
Insurellm offers health insurance to all full-time employees.
"""

text_b = """
Insurellm uses Jira to track engineering and security issues.


"""


probability = should_merge(text_a, text_b)

print("Probability of same chunk:", probability)

Probability of same chunk: 0.07


In [3]:
# Test 3
text_a = """
Insurellm offers health insurance to all full-time employees.
"""

text_b = """
Employees can participate in the company's 401(k) retirement plan.
"""


probability = should_merge(text_a, text_b)

print("Probability of same chunk:", probability)

Probability of same chunk: 0.43


In [ ]:
# Our first Large Jev Splitters
def split_paragraphs(paragraphs, threshold=0.75):

    chunks = []
    current_chunk = []

    for i, paragraph in enumerate(paragraphs):

        current_chunk.append(paragraph)

        # Last paragraph → finish the chunk
        if i == len(paragraphs) - 1:
            chunks.append("\n\n".join(current_chunk))
            break

        probability = should_merge(
            paragraph,
            paragraphs[i + 1]
        )

        print(
            f"Boundary {i + 1}: "
            f"{probability:.2f}"
        )

        if probability < threshold:
            chunks.append("\n\n".join(current_chunk))
            current_chunk = []

    return chunks
